# Fine-Tuning Pipeline
**ContextFlow AI** - Supervised Fine-Tuning dengan LoRA

1. Load preprocessed dataset
2. Load base model (TinyLlama)
3. Konfigurasi LoRA
4. Training dengan SFTTrainer
5. Save model fine-tuned

In [1]:
import sys
sys.path.append('..')

import torch
import os
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from app.utils.config import config

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Base model: {config.BASE_MODEL}')

d:\bootcamp\final_project_fine_tuning\contextflow_ai_fine_tuning\venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0516 09:21:51.894000 10212 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


PyTorch: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 2050
Base model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 1. Load Preprocessed Data

In [2]:
train_ds = load_from_disk(os.path.join(config.FORMATTED_DATA_DIR, 'train'))
val_ds = load_from_disk(os.path.join(config.FORMATTED_DATA_DIR, 'val'))

print(f'Train: {len(train_ds)} samples')
print(f'Val: {len(val_ds)} samples')
print(f'Columns: {train_ds.column_names}')
print()
print('Sample text:')
print(train_ds[0]['text'][:500])

Train: 34674 samples
Val: 3853 samples
Columns: ['instruction', 'input', 'output', 'text', 'source']

Sample text:
### Instruction:
When did Virgin Australia start operating?

### Input:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to d


In [3]:
import torch
print(torch.cuda.is_available())

True


## 2. Load Tokenizer

In [4]:
tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Vocab size: {tokenizer.vocab_size}')
print(f'Pad token: {tokenizer.pad_token}')

sample = train_ds[0]['text']
tokens = tokenizer(sample, truncation=True, max_length=config.MAX_SEQ_LENGTH)
print(f'Sample token length: {len(tokens["input_ids"])}')

Vocab size: 32000
Pad token: </s>
Sample token length: 181


## 3. Load Base Model

In [5]:
is_cuda = torch.cuda.is_available()

if is_cuda:
    model = AutoModelForCausalLM.from_pretrained(
        config.BASE_MODEL,
        torch_dtype=torch.float16,
        device_map={"": 0},  
        trust_remote_code=True,
    )
    print('Loaded with fp16 on GPU')
else:
    model = AutoModelForCausalLM.from_pretrained(
        config.BASE_MODEL,
        torch_dtype=torch.float32,
        device_map="cpu",  
        trust_remote_code=True,
    )
    print('Loaded with fp32 on CPU')

print(f'Model parameters: {model.num_parameters():,}')

Loaded with fp16 on GPU
Model parameters: 1,100,048,384


## 4. Konfigurasi LoRA

In [6]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# Cast LoRA trainable params ke float32 biar tidak konflik dengan gradient scaler
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


## 5. Training Arguments

In [7]:
output_dir = config.OUTPUT_MODEL_DIR

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=config.NUM_EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=config.LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=False,          # ← matikan dulu
    bf16=False,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
    optim='adamw_torch',
    seed=42,
    dataloader_num_workers=0,   # ← tambah ini
    dataloader_pin_memory=False # ← tambah ini
)

print(f'Epochs: {training_args.num_train_epochs}')
print(f'Batch size: {training_args.per_device_train_batch_size}')
print(f'Learning rate: {training_args.learning_rate}')

Epochs: 3
Batch size: 1
Learning rate: 0.0002


## 6. Fine-Tuning

In [8]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=256,
    packing=False,
)

print('Trainer ready!')

Trainer ready!


d:\bootcamp\final_project_fine_tuning\contextflow_ai_fine_tuning\venv311\Lib\site-packages\huggingface_hub\utils\_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
d:\bootcamp\final_project_fine_tuning\contextflow_ai_fine_tuning\venv311\Lib\site-packages\transformers\training_args.py:1965: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
d:\bootcamp\final_project_fine_tuning\contextflow_ai_fine_tuning\venv311\Lib\site-packages\trl\trainer\sft_trainer.py:269: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
d:\bootcamp\final_project_fi

In [9]:
# Mulai training
trainer.train()

  0%|          | 1/13002 [00:01<6:57:56,  1.93s/it]

KeyboardInterrupt: 

## 7. Save Model

In [ ]:
# Jalankan di cell baru untuk diagnosa
for name, param in trainer.model.named_parameters():
    if param.device.type == "meta":
        print(f"META: {name}")
        break
else:
    print("Tidak ada meta tensor")

print("\nDevice map:")
print(trainer.model.hf_device_map if hasattr(trainer.model, 'hf_device_map') else "No device map")

Tidak ada meta tensor

Device map:
{'': 0}


In [ ]:
# ✅ Save hanya LoRA adapter (bukan full model) — aman untuk QLoRA
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f'Model saved to {output_dir}')
for f in os.listdir(output_dir):
    size = os.path.getsize(os.path.join(output_dir, f))
    print(f'  {f}: {size/1024/1024:.1f} MB')

d:\bootcamp\final_project_fine_tuning\contextflow_ai_fine_tuning\venv311\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model saved to ./models/contextflow-finetuned
  adapter_config.json: 0.0 MB
  adapter_model.safetensors: 17.2 MB
  checkpoint-5400: 0.0 MB
  checkpoint-800: 0.0 MB
  README.md: 0.0 MB
  special_tokens_map.json: 0.0 MB
  tokenizer.json: 1.8 MB
  tokenizer.model: 0.5 MB
  tokenizer_config.json: 0.0 MB


## 8. Quick Inference Test

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL,
    device_map='auto' if is_cuda else None,
    torch_dtype=torch.float16 if is_cuda else torch.float32,
)
finetuned = PeftModel.from_pretrained(base, output_dir)
finetuned.eval()

prompt = '### Instruction:\nApa prosedur pengajuan cuti tahunan?\n\n### Input:\nKaryawan harus mengisi form HR-01.\n\n### Response:\n'
inputs = tokenizer(prompt, return_tensors='pt').to(finetuned.device)

with torch.no_grad():
    output = finetuned.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True, top_p=0.9)

result = tokenizer.decode(output[0], skip_special_tokens=True)
print('Generated:')
print(result[len(prompt):])

d:\bootcamp\final_project_fine_tuning\contextflow_ai_fine_tuning\venv311\Lib\site-packages\accelerate\utils\modeling.py:1365: UserWarning: Current model requires 1408 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
d:\bootcamp\final_project_fine_tuning\contextflow_ai_fine_tuning\venv311\Lib\site-packages\accelerate\utils\modeling.py:1365: UserWarning: Current model requires 2816 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Generated:
Apa prosedur pengajuan cuti tahunan? Karyawan harus mengisi form HR-01. Jika karyawan mengapa tidak ada form HR-01, maka karyawan bisa mengajukan cuti tahunan di kantor yang paling tinggi. Untuk mengajukan cuti tahunan di kantor yang paling tinggi, karyawan dapat mengajukan cuti tahunan di Kantor Pelayanan Penghuni (KPP) di kantor yang paling tinggi. Karyawan dapat mengajukan cuti tahunan di KPP dengan menyertai form HR-01 dan memasukkan form HR-01 dalam form
